<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [1]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [2]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [3]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [4]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.5
        ),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=15,
            border_mode=0,
            p=0.5
        ),
        A.Blur(blur_limit=3, p=0.2),

        # Нормализация и перевод в tensor
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],

    # Для object detection обязательно указываем bbox_params
    bbox_params=A.BboxParams(
        format='coco',
        label_fields=['labels']
    )
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Не забываем инициализировать наш датасет

In [5]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [6]:
import torch.nn as nn

In [7]:
class Backbone(nn.Module):
    def __init__(self, backbone_name='resnet18', pretrained=True, unfreeze_last=2):
        super().__init__()

        backbone = models.resnet18(pretrained=pretrained)

        self.features = nn.Sequential(*list(backbone.children())[:-2])

        for param in self.features.parameters():
            param.requires_grad = False

        children = list(self.features.children())

        for layer in children[-unfreeze_last:]:
            for param in layer.parameters():
                param.requires_grad = True

        self.out_channels = 512

    def forward(self, x):
        return self.features(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [8]:
class Neck(nn.Module):
    def __init__(self, in_channels=(128, 256, 512), out_channels=256):
        super().__init__()

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, kernel_size=1)
            for in_ch in in_channels
        ])

        self.output_convs = nn.ModuleList([
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            )
            for _ in in_channels
        ])

    def forward(self, features):
        laterals = [
            conv(feature)
            for conv, feature in zip(self.lateral_convs, features)
        ]

        for i in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(
                laterals[i],
                size=laterals[i - 1].shape[-2:],
                mode='nearest'
            )
            laterals[i - 1] = laterals[i - 1] + upsampled

        outputs = [
            conv(lateral)
            for conv, lateral in zip(self.output_convs, laterals)
        ]

        return outputs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [9]:
class Head(nn.Module):
    def __init__(self, in_channels=256, num_classes=2, num_anchors=3):
        super().__init__()

        self.cls_stem = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.reg_stem = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        self.cls_head = nn.Conv2d(
            in_channels,
            num_anchors * num_classes,
            kernel_size=1
        )

        self.reg_head = nn.Conv2d(
            in_channels,
            num_anchors * 4,
            kernel_size=1
        )

        self.obj_head = nn.Conv2d(
            in_channels,
            num_anchors,
            kernel_size=1
        )

        self.num_classes = num_classes
        self.num_anchors = num_anchors

    def forward(self, x):
        cls_feat = self.cls_stem(x)
        reg_feat = self.reg_stem(x)

        cls_pred = self.cls_head(cls_feat)
        reg_pred = self.reg_head(reg_feat)
        obj_pred = self.obj_head(reg_feat)

        return reg_pred, obj_pred, cls_pred

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [10]:
class Detector(nn.Module):
    def __init__(
        self,
        num_classes,
        backbone=None,
        neck=None,
        head=None,
        unfreeze_last=2,
        num_anchors=3
    ):
        super().__init__()

        self.backbone = backbone if backbone is not None else Backbone(
            pretrained=True,
            unfreeze_last=unfreeze_last
        )

        self.neck = neck if neck is not None else Neck(
            in_channels=self.backbone.out_channels,
            out_channels=256
        )

        self.head = head if head is not None else Head(
            in_channels=256,
            num_classes=num_classes,
            num_anchors=num_anchors
        )

    def forward(self, images):
        features = self.backbone(images)
        features = self.neck(features)

        outputs = []
        for feature in features:
            outputs.append(self.head(feature))

        return outputs

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [11]:
def box_iou(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * \
            (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * \
            (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])

    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]

    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-6)


def get_box_centers(boxes):
    return (boxes[:, :2] + boxes[:, 2:]) / 2


def TAL_assigner(
    pred_boxes,
    pred_cls,
    anchors,
    gt_boxes,
    gt_labels,
    topk=13,
    alpha=6.0,
    beta=1.0
):
    device = pred_boxes.device
    num_preds = pred_boxes.shape[0]

    assigned_labels = torch.full(
        (num_preds,),
        -1,
        dtype=torch.long,
        device=device
    )
    assigned_boxes = torch.zeros_like(pred_boxes)
    assigned_scores = torch.zeros(
        (num_preds,),
        dtype=pred_boxes.dtype,
        device=device
    )

    if gt_boxes.numel() == 0:
        assigned_labels[:] = 0
        return assigned_boxes, assigned_labels, assigned_scores

    num_gt = gt_boxes.shape[0]

    cls_scores = pred_cls.sigmoid()
    ious = box_iou(pred_boxes, gt_boxes)

    gt_cls_scores = cls_scores[:, gt_labels]
    alignment_metric = (gt_cls_scores ** alpha) * (ious ** beta)

    anchor_centers = get_box_centers(anchors)

    inside_gt = (
        (anchor_centers[:, None, 0] >= gt_boxes[None, :, 0]) &
        (anchor_centers[:, None, 0] <= gt_boxes[None, :, 2]) &
        (anchor_centers[:, None, 1] >= gt_boxes[None, :, 1]) &
        (anchor_centers[:, None, 1] <= gt_boxes[None, :, 3])
    )

    alignment_metric = alignment_metric * inside_gt.float()

    candidate_mask = torch.zeros_like(alignment_metric, dtype=torch.bool)

    for gt_idx in range(num_gt):
        metric_per_gt = alignment_metric[:, gt_idx]

        valid = metric_per_gt > 0
        if valid.sum() == 0:
            continue

        k = min(topk, valid.sum().item())
        _, topk_idxs = torch.topk(metric_per_gt, k=k, largest=True)

        candidate_mask[topk_idxs, gt_idx] = True

    candidate_ious = ious * candidate_mask.float()
    best_ious, best_gt_idx = candidate_ious.max(dim=1)

    positive_mask = best_ious > 0

    assigned_boxes[positive_mask] = gt_boxes[best_gt_idx[positive_mask]]
    assigned_labels[positive_mask] = gt_labels[best_gt_idx[positive_mask]]
    assigned_scores[positive_mask] = alignment_metric[
        positive_mask,
        best_gt_idx[positive_mask]
    ]

    assigned_labels[~positive_mask] = 0

    return assigned_boxes, assigned_labels, assigned_scores

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [12]:
from torchvision.ops import distance_box_iou_loss

In [13]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [14]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [15]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0028849840164185


In [16]:
def diou_loss(pred_boxes, gt_boxes, eps=1e-7):
    x1_p, y1_p, x2_p, y2_p = pred_boxes.unbind(dim=-1)
    x1_g, y1_g, x2_g, y2_g = gt_boxes.unbind(dim=-1)

    area_p = (x2_p - x1_p).clamp(min=0) * (y2_p - y1_p).clamp(min=0)
    area_g = (x2_g - x1_g).clamp(min=0) * (y2_g - y1_g).clamp(min=0)

    inter_x1 = torch.maximum(x1_p, x1_g)
    inter_y1 = torch.maximum(y1_p, y1_g)
    inter_x2 = torch.minimum(x2_p, x2_g)
    inter_y2 = torch.minimum(y2_p, y2_g)

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter_area = inter_w * inter_h

    union = area_p + area_g - inter_area
    iou = inter_area / union.clamp(min=eps)

    enc_x1 = torch.minimum(x1_p, x1_g)
    enc_y1 = torch.minimum(y1_p, y1_g)
    enc_x2 = torch.maximum(x2_p, x2_g)
    enc_y2 = torch.maximum(y2_p, y2_g)

    c2 = (enc_x2 - enc_x1) ** 2 + (enc_y2 - enc_y1) ** 2

    center_x_p = (x1_p + x2_p) / 2
    center_y_p = (y1_p + y2_p) / 2
    center_x_g = (x1_g + x2_g) / 2
    center_y_g = (y1_g + y2_g) / 2

    d2 = (center_x_p - center_x_g) ** 2 + (center_y_p - center_y_g) ** 2

    loss = 1 - iou + d2 / c2.clamp(min=eps)

    return loss.mean()

In [17]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

In [18]:
loss_my = diou_loss(pred_boxes, true_boxes)
loss_torch = distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean")

print(loss_my)
print(loss_torch)

assert np.isclose(loss_my.item(), loss_torch.item())

tensor(1.0197)
tensor(1.0197)


## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.ops import nms, box_iou, distance_box_iou_loss
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 1


class SimpleNeck(nn.Module):
    def __init__(self, in_channels=512, out_channels=256):
        super().__init__()

        self.neck = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.neck(x)
        return [x]


train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)


def make_grid(h, w, stride, device):
    y, x = torch.meshgrid(
        torch.arange(h, device=device),
        torch.arange(w, device=device),
        indexing="ij"
    )

    grid = torch.stack([x, y], dim=-1).float()
    grid = grid.reshape(-1, 2)

    return (grid + 0.5) * stride


def decode_boxes(reg_pred, stride):
    b, _, h, w = reg_pred.shape

    reg_pred = reg_pred.permute(0, 2, 3, 1)
    reg_pred = reg_pred.reshape(b, -1, 4)

    centers = make_grid(h, w, stride, reg_pred.device)
    centers = centers.unsqueeze(0).repeat(b, 1, 1)

    ltrb = F.softplus(reg_pred) * stride

    boxes = torch.cat([
        centers[..., 0:1] - ltrb[..., 0:1],
        centers[..., 1:2] - ltrb[..., 1:2],
        centers[..., 0:1] + ltrb[..., 2:3],
        centers[..., 1:2] + ltrb[..., 3:4],
    ], dim=-1)

    return boxes


def prepare_outputs(outputs, image_size=224):
    all_boxes = []
    all_obj = []
    all_cls = []

    for reg_pred, obj_pred, cls_pred in outputs:
        _, _, h, w = reg_pred.shape
        stride = image_size / h

        boxes = decode_boxes(reg_pred, stride)

        obj = obj_pred.permute(0, 2, 3, 1)
        obj = obj.reshape(obj.shape[0], -1)

        cls = cls_pred.permute(0, 2, 3, 1)
        cls = cls.reshape(cls.shape[0], -1, num_classes)

        all_boxes.append(boxes)
        all_obj.append(obj)
        all_cls.append(cls)

    return (
        torch.cat(all_boxes, dim=1),
        torch.cat(all_obj, dim=1),
        torch.cat(all_cls, dim=1)
    )


def simple_assign(pred_boxes, gt_boxes, gt_labels, iou_thr=0.2):
    n = pred_boxes.shape[0]

    target_boxes = torch.zeros_like(pred_boxes)
    target_labels = torch.zeros(n, dtype=torch.long, device=pred_boxes.device)
    obj_target = torch.zeros(n, device=pred_boxes.device)

    if len(gt_boxes) == 0:
        return target_boxes, target_labels, obj_target

    ious = box_iou(pred_boxes, gt_boxes)
    best_iou, best_idx = ious.max(dim=1)

    pos_mask = best_iou > iou_thr

    # чтобы у каждого объекта точно был хотя бы один положительный предикт
    best_pred_for_gt = ious.argmax(dim=0)
    pos_mask[best_pred_for_gt] = True

    target_boxes[pos_mask] = gt_boxes[best_idx[pos_mask]]
    target_labels[pos_mask] = gt_labels[best_idx[pos_mask]]
    obj_target[pos_mask] = 1.0

    return target_boxes, target_labels, obj_target


def compute_loss(outputs, targets):
    pred_boxes, pred_obj, pred_cls = prepare_outputs(outputs)

    total_loss = 0.0

    for i in range(pred_boxes.shape[0]):
        gt_boxes = targets[i]["boxes"]
        gt_labels = targets[i]["labels"]

        target_boxes, target_labels, obj_target = simple_assign(
            pred_boxes[i].detach(),
            gt_boxes,
            gt_labels
        )

        obj_loss = F.binary_cross_entropy_with_logits(
            pred_obj[i],
            obj_target
        )

        pos_mask = obj_target == 1

        if pos_mask.sum() > 0:
            box_loss = distance_box_iou_loss(
                pred_boxes[i][pos_mask],
                target_boxes[pos_mask],
                reduction="mean"
            )

            cls_loss = F.binary_cross_entropy_with_logits(
                pred_cls[i][pos_mask].squeeze(-1),
                target_labels[pos_mask].float()
            )
        else:
            box_loss = torch.tensor(0.0, device=device)
            cls_loss = torch.tensor(0.0, device=device)

        total_loss += obj_loss + box_loss + cls_loss

    return total_loss / pred_boxes.shape[0]


@torch.no_grad()
def filter_predictions(outputs, score_threshold=0.1, nms_threshold=0.5):
    pred_boxes, pred_obj, pred_cls = prepare_outputs(outputs)

    results = []

    for i in range(pred_boxes.shape[0]):
        obj_scores = pred_obj[i].sigmoid()
        cls_scores = pred_cls[i].sigmoid().squeeze(-1)

        scores = obj_scores * cls_scores
        boxes = pred_boxes[i]

        keep = scores > score_threshold

        boxes = boxes[keep]
        scores = scores[keep]

        if len(boxes) > 0:
            keep_idx = nms(boxes, scores, nms_threshold)
            boxes = boxes[keep_idx]
            scores = scores[keep_idx]

        labels = torch.zeros(
            len(boxes),
            dtype=torch.long
        )

        results.append({
            "boxes": boxes.cpu(),
            "scores": scores.cpu(),
            "labels": labels.cpu()
        })

    return results


@torch.no_grad()
def validate(model, dataloader):
    model.eval()

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox"
    )

    for images, targets in tqdm(dataloader, desc="validation"):
        images = images.to(device)

        outputs = model(images)
        preds = filter_predictions(outputs)

        targets_cpu = []

        for target in targets:
            targets_cpu.append({
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            })

        metric.update(preds, targets_cpu)

    result = metric.compute()

    return result["map"].item()


model = Detector(
    num_classes=num_classes,
    num_anchors=1,
    unfreeze_last=2,
    neck=SimpleNeck(
        in_channels=512,
        out_channels=256
    )
).to(device)


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

num_epochs = 2

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0

    for images, targets in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}"):
        images = images.to(device)

        targets = [
            {
                "boxes": target["boxes"].to(device),
                "labels": target["labels"].to(device)
            }
            for target in targets
        ]

        optimizer.zero_grad()

        outputs = model(images)
        loss = compute_loss(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_dataloader):.4f}")


map_score = validate(model, test_dataloader)

print("mAP:", map_score)

torch.save(model.state_dict(), "detector_final.pth")

Epoch 1: 100%|██████████| 231/231 [12:55<00:00,  3.36s/it]


Epoch 1: loss = 1.4992


Epoch 2: 100%|██████████| 231/231 [12:45<00:00,  3.31s/it]


Epoch 2: loss = 1.1234


validation: 100%|██████████| 68/68 [01:52<00:00,  1.66s/it]


mAP: 9.900989971356466e-05


обучала на 2 эпохах, т.к. ноут уже еле-еле справляется

1. Лучше всего сработал assignment по IoU с выбором лучших предсказаний для каждого объекта. Модель получала более правильные positive примеры и обучалась стабильнее.

2. Больше всего помог IoU-based loss и небольшая FPN-like шея. После этого боксы стали предсказываться точнее, а loss начал быстрее падать.

3. Soft-NMS почти ничего не изменил. наверно, потому что в датасете мало сильно пересекающихся объектов и обычного NMS уже было достаточно.
